In [7]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Set the project path

In [8]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/project-1/"
    "mango-disease-prediction"
)

MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

print("Project exists:", PROJECT_DIR.exists())
print("Models exists:", MODEL_DIR.exists())
print("Results exists:", RESULTS_DIR.exists())

Project exists: True
Models exists: True
Results exists: True


In [9]:
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("✅ Results directory ready")

✅ Results directory ready


In [10]:
print("ViT model:")
print(
    MODEL_DIR / "vit_image_only.pth",
    (MODEL_DIR / "vit_image_only.pth").exists()
)

print("\nMultimodal model:")
print(
    MODEL_DIR / "full_multimodal" / "best_model.pth",
    (MODEL_DIR / "full_multimodal" / "best_model.pth").exists()
)

ViT model:
/content/drive/MyDrive/project-1/mango-disease-prediction/models/vit_image_only.pth True

Multimodal model:
/content/drive/MyDrive/project-1/mango-disease-prediction/models/full_multimodal/best_model.pth True


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

PyTorch: 2.11.0+cpu
CUDA available: False
Device: cpu


In [12]:
class_names = [
    "Anthracnose",
    "Bacterial Canker",
    "Cutting Weevil",
    "Die Back",
    "Gall Midge",
    "Healthy",
    "Powdery Mildew",
    "Sooty Mould"
]

NUM_CLASSES = 8

print(class_names)

['Anthracnose', 'Bacterial Canker', 'Cutting Weevil', 'Die Back', 'Gall Midge', 'Healthy', 'Powdery Mildew', 'Sooty Mould']


Create the ViT architecture

In [13]:
from torchvision.models import vit_b_16

vit_model = vit_b_16(
    weights=None
)

vit_model.heads.head = nn.Linear(
    vit_model.heads.head.in_features,
    NUM_CLASSES
)

vit_model = vit_model.to(device)

print("✅ ViT architecture created")

✅ ViT architecture created


In [14]:
vit_path = MODEL_DIR / "vit_image_only.pth"

vit_model.load_state_dict(
    torch.load(
        vit_path,
        map_location=device,
        weights_only=True
    )
)

vit_model.eval()

print("✅ Trained ViT loaded")

✅ Trained ViT loaded


Weather Encoder

In [15]:
class WeatherEncoder(nn.Module):

    def __init__(
        self,
        input_dim=3,
        embedding_dim=768
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, embedding_dim)
        )

    def forward(self, x):

        return self.encoder(x).unsqueeze(1)


weather_encoder = WeatherEncoder(
    input_dim=3,
    embedding_dim=768
).to(device)

print("✅ Weather Encoder ready")

✅ Weather Encoder ready


Soil Encoder

In [16]:
class SoilEncoder(nn.Module):

    def __init__(
        self,
        input_dim=2,
        embedding_dim=768
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, embedding_dim)
        )

    def forward(self, x):

        return self.encoder(x).unsqueeze(1)


soil_encoder = SoilEncoder(
    input_dim=2,
    embedding_dim=768
).to(device)

print("✅ Soil Encoder ready")

✅ Soil Encoder ready


Season Encoder

In [17]:
class SeasonEncoder(nn.Module):

    def __init__(
        self,
        num_seasons=4,
        embedding_dim=768
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_seasons,
            embedding_dim
        )

    def forward(self, x):

        return self.embedding(x).unsqueeze(1)


season_encoder = SeasonEncoder(
    num_seasons=4,
    embedding_dim=768
).to(device)

print("✅ Season Encoder ready")

✅ Season Encoder ready


Cross-Attention

In [18]:
class CrossAttention(nn.Module):

    def __init__(
        self,
        embedding_dim=768,
        num_heads=8,
        dropout=0.1
    ):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(
            embedding_dim
        )

    def forward(
        self,
        image_features,
        environmental_features
    ):

        attended_features, attention_weights = (
            self.attention(
                image_features,
                environmental_features,
                environmental_features
            )
        )

        output = self.norm(
            image_features + attended_features
        )

        return output, attention_weights


cross_attention = CrossAttention(
    embedding_dim=768,
    num_heads=8,
    dropout=0.1
).to(device)

print("✅ Cross-Attention ready")

✅ Cross-Attention ready


Check the components

In [19]:
print("Weather Encoder:", weather_encoder)
print("Soil Encoder:", soil_encoder)
print("Season Encoder:", season_encoder)
print("Cross Attention:", cross_attention)

Weather Encoder: WeatherEncoder(
  (encoder): Sequential(
    (0): Linear(in_features=3, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=768, bias=True)
  )
)
Soil Encoder: SoilEncoder(
  (encoder): Sequential(
    (0): Linear(in_features=2, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=768, bias=True)
  )
)
Season Encoder: SeasonEncoder(
  (embedding): Embedding(4, 768)
)
Cross Attention: CrossAttention(
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


Recreate the full model

In [21]:
class FullMangoMultimodalModel(nn.Module):

    def __init__(
        self,
        vit,
        num_seasons=4,
        num_classes=8,
        embedding_dim=768
    ):
        super().__init__()

        # -----------------------------
        # ViT
        # -----------------------------
        self.vit = vit

        # -----------------------------
        # Environmental encoders
        # -----------------------------
        self.weather_encoder = WeatherEncoder(
            input_dim=3,
            embedding_dim=embedding_dim
        )

        self.soil_encoder = SoilEncoder(
            input_dim=2,
            embedding_dim=embedding_dim
        )

        self.season_encoder = SeasonEncoder(
            num_seasons=num_seasons,
            embedding_dim=embedding_dim
        )

        # -----------------------------
        # Cross Attention
        # -----------------------------
        self.cross_attention = CrossAttention(
            embedding_dim=embedding_dim,
            num_heads=8,
            dropout=0.1
        )

        # -----------------------------
        # Fusion Transformer
        # -----------------------------
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=8,
            dim_feedforward=embedding_dim * 4,
            dropout=0.1,
            batch_first=True
        )

        self.fusion_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.fusion_norm = nn.LayerNorm(
            embedding_dim
        )

        # -----------------------------
        # Classifier
        # -----------------------------
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(
        self,
        images,
        weather,
        soil,
        season
    ):

        # ViT features
        image_features = self.vit._process_input(images)

        n = image_features.shape[0]

        batch_class_token = (
            self.vit.class_token
            .expand(n, -1, -1)
        )

        image_features = torch.cat(
            [batch_class_token, image_features],
            dim=1
        )

        image_features = image_features + \
            self.vit.encoder.pos_embedding

        image_features = self.vit.encoder(
            image_features
        )

        # Environmental tokens
        weather_token = self.weather_encoder(
            weather
        )

        soil_token = self.soil_encoder(
            soil
        )

        season_token = self.season_encoder(
            season
        )

        environmental_features = torch.cat(
            [
                weather_token,
                soil_token,
                season_token
            ],
            dim=1
        )

        # Cross attention
        attended_features, attention_weights = (
            self.cross_attention(
                image_features,
                environmental_features
            )
        )

        # Fusion Transformer
        fusion_features = self.fusion_transformer(
            attended_features
        )

        fusion_features = self.fusion_norm(
            fusion_features
        )

        # Use CLS token
        joint_features = fusion_features[:, 0]

        # Classification
        logits = self.classifier(
            joint_features
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        confidence = torch.max(
            probabilities,
            dim=1
        ).values

        return {
            "logits": logits,
            "probabilities": probabilities,
            "predictions": predictions,
            "confidence": confidence,
            "attention_weights": attention_weights,
            "fusion_features": fusion_features
        }


print("✅ Full model class created")

✅ Full model class created


Create the model

In [22]:
full_mango_model = FullMangoMultimodalModel(
    vit=vit_model,
    num_seasons=4,
    num_classes=8,
    embedding_dim=768
).to(device)

print("✅ Full Mango model created")

✅ Full Mango model created


Load your trained model

In [23]:
best_model_path = (
    MODEL_DIR /
    "full_multimodal" /
    "best_model.pth"
)

checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False
)

print(type(checkpoint))

<class 'dict'>


In [24]:
best_model_path = (
    MODEL_DIR /
    "full_multimodal" /
    "best_model.pth"
)

checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False
)

print(type(checkpoint))

<class 'dict'>


In [26]:
state_dict = checkpoint["model_state_dict"]

print("=" * 60)
print("CHECKPOINT ARCHITECTURE")
print("=" * 60)

for key, value in state_dict.items():

    print(
        f"{key:65s} {tuple(value.shape)}"
    )

CHECKPOINT ARCHITECTURE
vit.class_token                                                   (1, 1, 768)
vit.conv_proj.weight                                              (768, 3, 16, 16)
vit.conv_proj.bias                                                (768,)
vit.encoder.pos_embedding                                         (1, 197, 768)
vit.encoder.layers.encoder_layer_0.ln_1.weight                    (768,)
vit.encoder.layers.encoder_layer_0.ln_1.bias                      (768,)
vit.encoder.layers.encoder_layer_0.self_attention.in_proj_weight  (2304, 768)
vit.encoder.layers.encoder_layer_0.self_attention.in_proj_bias    (2304,)
vit.encoder.layers.encoder_layer_0.self_attention.out_proj.weight (768, 768)
vit.encoder.layers.encoder_layer_0.self_attention.out_proj.bias   (768,)
vit.encoder.layers.encoder_layer_0.ln_2.weight                    (768,)
vit.encoder.layers.encoder_layer_0.ln_2.bias                      (768,)
vit.encoder.layers.encoder_layer_0.mlp.0.weight                   (3

In [27]:
class WeatherEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 256)
        )

    def forward(self, x):
        return self.encoder(x)


class SoilEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 256)
        )

    def forward(self, x):
        return self.encoder(x)


class SeasonEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            2,
            256
        )

    def forward(self, x):
        return self.embedding(x)


class CrossAttention(nn.Module):

    def __init__(
        self,
        embedding_dim=768,
        num_heads=8,
        dropout=0.1
    ):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(
            embedding_dim
        )

    def forward(
        self,
        image_features,
        environmental_features
    ):

        attended, attention_weights = self.attention(
            image_features,
            environmental_features,
            environmental_features
        )

        output = self.norm(
            image_features + attended
        )

        return output, attention_weights


print("✅ Exact encoders created")

✅ Exact encoders created


In [28]:
class FusionTransformer(nn.Module):

    def __init__(
        self,
        embedding_dim=768,
        num_heads=8,
        num_layers=2,
        dropout=0.1
    ):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=2048,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(
            embedding_dim
        )

    def forward(self, x):

        x = self.transformer(x)

        x = self.norm(x)

        return x


print("✅ Exact Fusion Transformer created")

✅ Exact Fusion Transformer created


In [29]:
class FullMangoMultimodalModel(nn.Module):

    def __init__(
        self,
        vit,
        num_classes=8
    ):
        super().__init__()

        self.vit = vit

        self.weather_encoder = WeatherEncoder()

        self.soil_encoder = SoilEncoder()

        self.season_encoder = SeasonEncoder()

        self.environment_projection = nn.Linear(
            256,
            768
        )

        self.cross_attention = CrossAttention(
            embedding_dim=768,
            num_heads=8,
            dropout=0.1
        )

        self.fusion_transformer = FusionTransformer(
            embedding_dim=768,
            num_heads=8,
            num_layers=2,
            dropout=0.1
        )

        self.disease_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(
        self,
        images,
        weather,
        soil,
        season
    ):

        # ==================================
        # ViT IMAGE FEATURES
        # ==================================

        image_features = self.vit._process_input(
            images
        )

        n = image_features.shape[0]

        batch_class_token = (
            self.vit.class_token.expand(
                n, -1, -1
            )
        )

        image_features = torch.cat(
            [
                batch_class_token,
                image_features
            ],
            dim=1
        )

        image_features = (
            image_features +
            self.vit.encoder.pos_embedding
        )

        image_features = self.vit.encoder(
            image_features
        )

        # ==================================
        # ENVIRONMENT
        # ==================================

        weather_features = (
            self.weather_encoder(weather)
        )

        soil_features = (
            self.soil_encoder(soil)
        )

        season_features = (
            self.season_encoder(season)
        )

        # [B, 3, 256]
        environmental_features = torch.stack(
            [
                weather_features,
                soil_features,
                season_features
            ],
            dim=1
        )

        # [B, 3, 768]
        environmental_features = (
            self.environment_projection(
                environmental_features
            )
        )

        # ==================================
        # CROSS ATTENTION
        # ==================================

        attended_features, attention_weights = (
            self.cross_attention(
                image_features,
                environmental_features
            )
        )

        # ==================================
        # FUSION TRANSFORMER
        # ==================================

        fusion_features = (
            self.fusion_transformer(
                attended_features
            )
        )

        # ==================================
        # CLS TOKEN
        # ==================================

        joint_features = fusion_features[:, 0]

        # ==================================
        # DISEASE HEAD
        # ==================================

        logits = self.disease_head(
            joint_features
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        confidence = torch.max(
            probabilities,
            dim=1
        ).values

        return {
            "logits": logits,
            "probabilities": probabilities,
            "predictions": predictions,
            "confidence": confidence,
            "attention_weights": attention_weights,
            "fusion_features": fusion_features
        }


print("✅ EXACT FULL MANGO MODEL CREATED")

✅ EXACT FULL MANGO MODEL CREATED


In [30]:
full_mango_model = FullMangoMultimodalModel(
    vit=vit_model,
    num_classes=8
).to(device)

print("✅ Model instance created")

✅ Model instance created


Load best_model.pth

In [31]:
best_model_path = (
    MODEL_DIR
    / "full_multimodal"
    / "best_model.pth"
)

checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded")
print("Checkpoint keys:", checkpoint.keys())

Checkpoint loaded
Checkpoint keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_accuracy'])


In [32]:
full_mango_model.load_state_dict(
    checkpoint["model_state_dict"]
)

full_mango_model.eval()

print("✅ EXACT TRAINED MODEL LOADED")

✅ EXACT TRAINED MODEL LOADED


In [34]:
test_csv = PROJECT_DIR / "data" / "processed" / "test.csv"

print("Test CSV exists:", test_csv.exists())
print(test_csv)

Test CSV exists: True
/content/drive/MyDrive/project-1/mango-disease-prediction/data/processed/test.csv


In [35]:
test_df = pd.read_csv(test_csv)

print("Shape:", test_df.shape)
print("\nColumns:")
print(test_df.columns.tolist())

print("\nFirst 5 rows:")
display(test_df.head())

Shape: (600, 14)

Columns:
['image_path', 'class_name', 'label', 'sample_index', 'sample_id', 'date', 'time', 'temperature_c', 'humidity_pct', 'rainfall_mm', 'soil_moisture_pct', 'soil_ph', 'season', 'data_status']

First 5 rows:


,image_path,class_name,label,sample_index,sample_id,date,time,temperature_c,humidity_pct,rainfall_mm,soil_moisture_pct,soil_ph,season,data_status
0,/content/drive/MyDrive/project-1/mango-disease...,Die Back,3,235,Die_Back_236,2021-10-08,09:17:19,30.13,85.78,3.69,67.32,6.22,Post-Monsoon,SIMULATED
1,/content/drive/MyDrive/project-1/mango-disease...,Healthy,5,13,Healthy_014,2021-11-22,09:24:30,28.03,75.06,0.00,69.98,6.91,Post-Monsoon,SIMULATED
2,/content/drive/MyDrive/project-1/mango-disease...,Powdery Mildew,6,115,Powdery_Mildew_116,2021-12-28,17:02:17,21.29,68.71,2.17,72.07,6.16,Winter,SIMULATED
3,/content/drive/MyDrive/project-1/mango-disease...,Healthy,5,272,Healthy_273,2021-11-14,15:38:18,22.40,74.35,0.00,61.52,6.56,Post-Monsoon,SIMULATED
4,/content/drive/MyDrive/project-1/mango-disease...,Powdery Mildew,6,33,Powdery_Mildew_034,2021-12-06,15:55:53,23.49,59.58,16.89,80.08,6.56,Winter,SIMULATED


In [36]:
test_csv = PROJECT_DIR / "data" / "processed" / "test.csv"

print("Test CSV exists:", test_csv.exists())
print("Path:", test_csv)

Test CSV exists: True
Path: /content/drive/MyDrive/project-1/mango-disease-prediction/data/processed/test.csv


In [37]:
test_df = pd.read_csv(test_csv)

print("Shape:", test_df.shape)
print("\nColumns:")
print(test_df.columns.tolist())

print("\nFirst 5 rows:")
display(test_df.head())

Shape: (600, 14)

Columns:
['image_path', 'class_name', 'label', 'sample_index', 'sample_id', 'date', 'time', 'temperature_c', 'humidity_pct', 'rainfall_mm', 'soil_moisture_pct', 'soil_ph', 'season', 'data_status']

First 5 rows:


,image_path,class_name,label,sample_index,sample_id,date,time,temperature_c,humidity_pct,rainfall_mm,soil_moisture_pct,soil_ph,season,data_status
0,/content/drive/MyDrive/project-1/mango-disease...,Die Back,3,235,Die_Back_236,2021-10-08,09:17:19,30.13,85.78,3.69,67.32,6.22,Post-Monsoon,SIMULATED
1,/content/drive/MyDrive/project-1/mango-disease...,Healthy,5,13,Healthy_014,2021-11-22,09:24:30,28.03,75.06,0.00,69.98,6.91,Post-Monsoon,SIMULATED
2,/content/drive/MyDrive/project-1/mango-disease...,Powdery Mildew,6,115,Powdery_Mildew_116,2021-12-28,17:02:17,21.29,68.71,2.17,72.07,6.16,Winter,SIMULATED
3,/content/drive/MyDrive/project-1/mango-disease...,Healthy,5,272,Healthy_273,2021-11-14,15:38:18,22.40,74.35,0.00,61.52,6.56,Post-Monsoon,SIMULATED
4,/content/drive/MyDrive/project-1/mango-disease...,Powdery Mildew,6,33,Powdery_Mildew_034,2021-12-06,15:55:53,23.49,59.58,16.89,80.08,6.56,Winter,SIMULATED


In [38]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import numpy as np

class MangoTestDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        # Season mapping used by the model
        seasons = sorted(
            self.df["season"].unique()
        )

        self.season_to_id = {
            season: i
            for i, season in enumerate(seasons)
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # -----------------------------
        # IMAGE
        # -----------------------------

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        image = self.transform(image)

        # -----------------------------
        # WEATHER
        # -----------------------------

        weather = torch.tensor(
            [
                row["temperature_c"],
                row["humidity_pct"],
                row["rainfall_mm"]
            ],
            dtype=torch.float32
        )

        # -----------------------------
        # SOIL
        # -----------------------------

        soil = torch.tensor(
            [
                row["soil_moisture_pct"],
                row["soil_ph"]
            ],
            dtype=torch.float32
        )

        # -----------------------------
        # SEASON
        # -----------------------------

        season = torch.tensor(
            self.season_to_id[row["season"]],
            dtype=torch.long
        )

        # -----------------------------
        # LABEL
        # -----------------------------

        label = torch.tensor(
            int(row["label"]),
            dtype=torch.long
        )

        return {
            "image": image,
            "weather": weather,
            "soil": soil,
            "season": season,
            "label": label
        }


print("✅ Test Dataset class created")

✅ Test Dataset class created


In [39]:
test_dataset = MangoTestDataset(test_df)

test_full_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✅ test_full_loader created")
print("Number of samples:", len(test_dataset))
print("Number of batches:", len(test_full_loader))

✅ test_full_loader created
Number of samples: 600
Number of batches: 38


In [40]:
print("Seasons in test data:")
print(test_df["season"].unique())

print("\nSeason mapping:")
print(test_dataset.season_to_id)

Seasons in test data:
['Post-Monsoon' 'Winter']

Season mapping:
{'Post-Monsoon': 0, 'Winter': 1}


Check season mapping

In [41]:
print("Seasons in test data:")
print(test_df["season"].unique())

print("\nSeason mapping:")
print(test_dataset.season_to_id)

Seasons in test data:
['Post-Monsoon' 'Winter']

Season mapping:
{'Post-Monsoon': 0, 'Winter': 1}


Check one batch

In [42]:
batch = next(iter(test_full_loader))

print("Images :", batch["image"].shape)
print("Weather:", batch["weather"].shape)
print("Soil   :", batch["soil"].shape)
print("Season :", batch["season"].shape)
print("Labels :", batch["label"].shape)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Images : torch.Size([16, 3, 224, 224])
Weather: torch.Size([16, 3])
Soil   : torch.Size([16, 2])
Season : torch.Size([16])
Labels : torch.Size([16])


Run the trained model

In [43]:
images = batch["image"].to(device)
weather = batch["weather"].to(device)
soil = batch["soil"].to(device)
season = batch["season"].to(device)
labels = batch["label"].to(device)

with torch.no_grad():

    result = full_mango_model(
        images,
        weather,
        soil,
        season
    )

print("Logits      :", result["logits"].shape)
print("Predictions :", result["predictions"].shape)
print("Confidence  :", result["confidence"].shape)
print("Attention   :", result["attention_weights"].shape)
print("Fusion      :", result["fusion_features"].shape)

Logits      : torch.Size([16, 8])
Predictions : torch.Size([16])
Confidence  : torch.Size([16])
Attention   : torch.Size([16, 197, 3])
Fusion      : torch.Size([16, 197, 768])


In [1]:
for i in range(5):

    true_label = labels[i].item()
    pred_label = result["predictions"][i].item()
    conf = result["confidence"][i].item()

    print(
        f"{i}: "
        f"True = {class_names[true_label]} | "
        f"Predicted = {class_names[pred_label]} | "
        f"Confidence = {conf * 100:.2f}%"
    )

NameError: name 'labels' is not defined